# MFCC + SVM

**Модель:** SVM (RBF) на статистиках MFCC (mean + std + delta_mean + delta_std)

In [62]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import make_scorer, f1_score

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix
from shared.results_utils import save_result_csv

In [63]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)} good: {np.sum(labels_trainval==0)}, bad: {np.sum(labels_trainval==1)}")
print(f"Test:      {len(paths_test)}  good: {np.sum(labels_test==0)},  bad: {np.sum(labels_test==1)}")

Train+Val: 2356 good: 1594, bad: 762
Test:      416  good: 281,  bad: 135


## Подбор гиперпараметров (GridSearchCV на train+val)

In [64]:
extractor = data_utils.extract_mfcc_stats

print("Извлечение признаков (train+val)...")
X_trainval_raw = build_feature_matrix(paths_trainval, extractor)
print("Извлечение признаков (test)...")
X_test_raw = build_feature_matrix(paths_test, extractor)

X_trainval = np.hstack([X_trainval_raw, letters_trainval])
X_test     = np.hstack([X_test_raw,     letters_test])


Извлечение признаков (train+val)...
Извлечение признаков (test)...


In [65]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

t0 = time.perf_counter()
grid = GridSearchCV(pipeline, param_grid, cv=5,
                    scoring=f1_bad_scorer, refit=True, n_jobs=-1)
grid.fit(X_trainval, labels_trainval)
train_time_sec = time.perf_counter() - t0

best_params = grid.best_params_
best_clf = grid.best_estimator_
print(f"Лучшие параметры: {best_params}")

Лучшие параметры: {'clf__C': 1.0, 'clf__gamma': 'scale'}


## Оценка на test

In [66]:
y_proba = cross_val_predict(
    best_clf, X_trainval, labels_trainval, cv=5, method="predict_proba"
)[:, config.CLASS_BAD]
threshold = find_optimal_threshold(labels_trainval, y_proba)

y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_mfcc_svm",
    experiment_name="MFCC + SVM (baseline)",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=80,
    embed_dim_note="MFCC 20 коэф. * 4 статистики (mean, std, delta_mean, delta_std)",
    notes=f"GridSearchCV cv=5 + threshold | best_params={best_params} | thr={threshold:.2f}",
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.83      0.74      0.78       281
         bad       0.55      0.68      0.61       135

    accuracy                           0.72       416
   macro avg       0.69      0.71      0.70       416
weighted avg       0.74      0.72      0.73       416

Threshold : 0.29
Accuracy  : 0.7188
F1-macro  : 0.6955
F1-bad    : 0.6113
ROC-AUC   : 0.7863


PosixPath('/Users/dk/Desktop/ВШЭ/ВКР/HSE_VKR_DetectingSpeechDefects/experiments/01_baselines/exp_mfcc_svm/result.csv')